In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import sklearn.ensemble as ske
import time
import numpy as np
import pynq
import math
import datetime
import struct
import asyncio
import os
import re
import ast
import sys
from converter import Converter
from MultiDepthRandomForest import MultiDepthNeuralRandomForestClassifier

In [ ]:
dataframes = []
for i in tqdm(range(15)):
    string = "./accelerometer/" + str(i+1) + ".csv"
    dataframes.append(pd.read_csv(string,header=None))
    dataframes[i].columns = ["pos","x","y","z","action"]

for i,dataframe in tqdm(enumerate(dataframes)):
    dataframes[i] = dataframe.drop("pos",axis=1)

In [ ]:
def calculate_params(matrix):
    params = []
    for i in range(matrix.shape[1] - 1):
        mean = np.mean(matrix[:,i])
        std = np.std(matrix[:,i])
        perc25 = np.percentile(matrix[:,i],25)
        perc50 = np.percentile(matrix[:,i],50)
        perc75 = np.percentile(matrix[:,i],75)
        f = np.fft.fft(matrix[:,i])
        spEnergy = np.sum(f**2)/len(f)
        params.append(mean)
        params.append(std)
        params.append(perc25)
        params.append(perc50)
        params.append(perc75)
        params.append(spEnergy)
    params.append(matrix[0,3])
    return params

In [ ]:
dataset = np.zeros((1,19))
for dataframe in tqdm(dataframes):
    matrix = dataframe.to_numpy()
    matrices = []
    matrices.append(matrix[matrix[:,3]==1])
    matrices.append(matrix[matrix[:,3]==2])
    matrices.append(matrix[matrix[:,3]==3])
    matrices.append(matrix[matrix[:,3]==4])
    matrices.append(matrix[matrix[:,3]==5])
    matrices.append(matrix[matrix[:,3]==6])
    matrices.append(matrix[matrix[:,3]==7])
    for i,matrix in enumerate(matrices):
        if i != 1:
            stop = False
            index = 0
            while(not stop):
                if index + 1000 > matrix.shape[0]:
                    sample = matrix[index:matrix.shape[0]]
                    if sample.shape[0] != 0:
                        features = calculate_params(sample)
                    stop = True
                else:
                    sample = matrix[index:index + 1000]
                    features = calculate_params(sample)
                index += 1000
                dataset = np.insert(dataset,dataset.shape[0],features,axis=0)
        else:
            sample = matrix
            features = calculate_params(sample)
            dataset = np.insert(dataset,dataset.shape[0],features,axis=0)

In [ ]:
dataset1 = np.delete(dataset,0,0)

In [ ]:
np.random.seed(1234) #for reproducibility
np.random.shuffle(dataset1)
n_samples = dataset1.shape[0]
X_train = dataset1[:int(0.75*n_samples),:18]
Y_train = dataset1[:int(0.75*n_samples),18]
X_val = dataset1[int(0.75*n_samples):int(0.90*n_samples),:18]
Y_val = dataset1[int(0.75*n_samples):int(0.90*n_samples),18]
X_test = dataset1[int(0.9*n_samples):,:18]
Y_test = dataset1[int(0.9*n_samples):,18]
Y_train -= 1
Y_val -= 1
Y_test -= 1

In [ ]:
X_train.shape, Y_train.shape, X_val.shape, Y_val.shape, X_test.shape, Y_test.shape

In [ ]:
mean = np.mean(X_train,axis=0)
std = np.std(X_train,axis=0)
X_train = (X_train - mean)/std
X_val = (X_val - mean)/std
X_test = (X_test - mean)/std

## Backward feature selection

In [ ]:
seed = 1234
n_features = 18
end = False
selected = []
while(not end):
    classifier = ske.RandomForestClassifier(n_estimators=100,max_depth=7, random_state=3)
    classifier.fit(X_train[:,selected], Y_train)
    scores = {}
    scores[-1] = classifier.score(X_val[:,selected],Y_val)
    print("Performance with all the remaining features: ", scores[-1])
    for value in selected:
        selected_this_round = selected.copy()
        selected_this_round.remove(value)
        classifier = ske.RandomForestClassifier(n_estimators=100,max_depth=7, random_state = 3)
        classifier.fit(X_train[:,selected_this_round], Y_train)
        scores[value] = classifier.score(X_val[:,selected_this_round],Y_val)
        print("Performance without the feature ", value, ": ", scores[value])
    
    stop = True
    for key in scores.keys():
        if scores[-1] < scores[key]:
            stop = False
        
    if not stop:
        keys = list(scores.keys())
        values = list(scores.values())
        best = np.argmax(values)
        to_remove = keys[best]
        selected.remove(to_remove)
        print("feature ", to_remove," deleted")
        print("Remaining features", selected)
    else:
        end = True
        print("No feature removed, process stopped")
        

### Forward feature selection

In [ ]:
seed = 1234
end = False
selected = []
to_select = list(np.arange(18))
while(not end):
    scores = {}
    if len(selected) != 0:
        classifier = ske.RandomForestClassifier(n_estimators=100,max_depth=7, random_state=3)
        classifier.fit(X_train[:,selected], Y_train)
        scores[-1] = classifier.score(X_val[:,selected],Y_val)
    else:
        scores[-1] = 0
        
    print("Performance with the starting features: ", scores[-1])
    for value in to_select:
        selected_this_round = selected.copy()
        selected_this_round.append(value)
        classifier = ske.RandomForestClassifier(n_estimators=100,max_depth=7, random_state = 3)
        classifier.fit(X_train[:,selected_this_round], Y_train)
        scores[value] = classifier.score(X_val[:,selected_this_round],Y_val)
        print("Performance adding the feature ", value, ": ", scores[value])
    
    stop = True
    for key in scores.keys():
        if scores[-1] < scores[key]:
            stop = False
        
    if not stop:
        keys = list(scores.keys())
        values = list(scores.values())
        best = np.argmax(values)
        to_add = keys[best]
        selected.append(to_add)
        print("feature ", to_add," added")
        print("selected features", selected)
    else:
        end = True
        print("No feature added, process stopped")
        

## If number of classes is different from 7

In [ ]:
n_classes = 7
X_train_reduced = X_train[Y_train<n_classes]
Y_train_reduced = Y_train[Y_train<n_classes]
X_test_reduced = X_test[Y_test<n_classes]
Y_test_reduced = Y_test[Y_test<n_classes]

In [ ]:
X_train_reduced.shape, Y_train_reduced.shape, X_test_reduced.shape, Y_test_reduced.shape

## TRAINING

In [ ]:
n_trees = 50
max_depth = 5
min_depth = 1
n_attr = 5
n_depths = max_depth - min_depth + 1
mdcls = MultiDepthNeuralRandomForestClassifier(total_estimators=n_trees,min_depth=min_depth,max_depth=max_depth,random_state=3)
mdcls.fit(X_train_reduced[:,:n_attr], Y_train_reduced)
mdcls.score(X_test_reduced[:,:n_attr],Y_test_reduced)

In [ ]:
class NOInst:
    def __init__(self, nodeRA, attr_id, threshold, lctype, lcinfo, rctype, rcinfo, layer=0, is_valid = True, depth_indicator = 0):
        self.nodeRA = nodeRA
        self.attr_id = attr_id
        self.threshold = threshold
        self.lctype = lctype
        self.lcinfo = lcinfo
        self.rctype = rctype
        self.rcinfo = rcinfo
        self.layer = layer 
        self.is_valid = is_valid 
        self.depth_indicator = depth_indicator
        
class Node:
    def __init__(self, nodeRA, ram_id, left_child, right_child, attr_id, threshold,layer):
        self.nodeRA = nodeRA
        self.ram_id = ram_id
        self.left_child = left_child
        self.right_child = right_child
        self.attr_id = attr_id
        self.threshold = threshold
        self.layer = layer
        
def transform_into_instructions(estimators_per_depth, N, M, N_tree, n_depths): #N max NOinsts in one BRAM, M maximum available BRAMs, N_tree number of tree contained in a BRAM
    estimators_per_depth = list(reversed(estimators_per_depth))
    FLInsts = np.zeros(M,dtype=np.int8)
    NOInsts = []
    current_id = 0
    estimators = []
    accumulate = 0
    exit = False
    to_add = {}
    accumulate = {}
    for i in range(n_depths):
        accumulate[i] = 0
    while not exit:
        for i in range(n_depths):
            to_add[i] = min(int(N_tree/n_depths),len(estimators_per_depth[i]) - accumulate[i])
         
        j = 0
        for cls in estimators_per_depth:
            for i in range(to_add[j]):
                estimators.append(cls[accumulate[j] + i])
            j += 1
            
        for i in range(n_depths):
            accumulate[i] += to_add[i]
        
        exit = True
        for i in range(n_depths):
            if accumulate[i] < len(estimators_per_depth[i]):
                exit = False
    
    for i, estimator in enumerate(estimators):
        depth = estimator.max_depth - min_depth
        order_in_bram = i%N_tree
        nodes = {}
        FLInsts[current_id] = 1
        this_level_nodes = []
        next_level_nodes = [0]
        for j in range(estimator.tree_.max_depth):
            this_level_nodes = next_level_nodes.copy()
            next_level_nodes = []
            needed_bram = int(len(this_level_nodes)/(N+1)) + 1
            for k,node in enumerate(this_level_nodes):
                if estimator.tree_.children_left[node] != -1 and estimator.tree_.children_right[node] != -1:
                    index = node
                    ram_id = current_id + int(k/N)
                    nodeRA = k%N + order_in_bram*(2**j)
                    left_children = estimator.tree_.children_left[node]
                    right_children = estimator.tree_.children_right[node]
                    attr_id = estimator.tree_.feature[node]
                    threshold = estimator.tree_.threshold[node]
                    nodes[node] = Node(nodeRA,ram_id,left_children,right_children,attr_id,threshold,j)
                    next_level_nodes.append(estimator.tree_.children_left[node])
                    next_level_nodes.append(estimator.tree_.children_right[node])
            current_id += needed_bram
        for node in nodes.values():
            if estimator.tree_.feature[node.left_child] < 0:
                left_child_type = 0
                left_child_info = np.argmax(estimator.tree_.value[node.left_child,0])
            else:
                left_child_type = nodes[node.left_child].ram_id - node.ram_id
                left_child_info = nodes[node.left_child].nodeRA
            if estimator.tree_.feature[node.right_child] < 0:
                right_child_type = 0
                right_child_info = np.argmax(estimator.tree_.value[node.right_child,0])
            else:
                right_child_type = nodes[node.right_child].ram_id - node.ram_id
                right_child_info = nodes[node.right_child].nodeRA
            NOInsts.append(NOInst(node.nodeRA,int(node.attr_id),node.threshold,left_child_type,left_child_info,right_child_type,right_child_info,node.layer,True,depth))
    return FLInsts, NOInsts

# Split in multiple paths

In [ ]:
n_paths = 5
trees_per_path = int(n_trees/n_paths) #assume at the moment that the total number of trees is multiple of n_paths
trees_per_path_per_depth = int(trees_per_path/n_depths)
max_trees_per_set = 36*1024//(64*(2**(max_depth-1)))*n_depths
max_inst = 576
NOInsts_list = []
FLInsts_list = []
for path in range(n_paths):
    estimators = []
    for k in range(n_depths):
        estimators.append(list(mdcls.classifiers.values())[k].estimators_[path*trees_per_path_per_depth:(path+1)*trees_per_path_per_depth])
    FLInsts_per_depth, NOInsts_per_depth = transform_into_instructions(estimators,max_inst,1000000,max_trees_per_set,n_depths)
    NOInsts_list.append(NOInsts_per_depth)
    FLInsts_list.append(FLInsts_per_depth)

n_topfs = int(trees_per_path/max_trees_per_set+1)*max_depth
if trees_per_path%max_trees_per_set == 0:
    n_topfs = int(n_trees/max_trees_per_set)*max_depth

for i,instr_set in enumerate(NOInsts_list):
    for j,inst in enumerate(instr_set):
        print(inst.__dict__)
    print(FLInsts_list[i][:n_topfs])

In [ ]:
class Sample_w_Score:
    def __init__(self, features , shift, offset, search_for_root, tree_to_exec, scores, weights):
        self.features = features
        self.shift = shift
        self.offset = offset
        self.search_for_root = search_for_root
        self.tree_to_exec = tree_to_exec
        self.scores = scores
        self.weights = weights

In [ ]:
class Accel: #32 bits attribute version

    def __init__(self, overlay, converter, platform='Ultra96_v2', n_brams=100, n_samples=1000, dim = 1000, n_classes=3, n_attr=5, n_weights=5, trees_in_bram=3, max_inst=576, frq_MHZ = 100):
        self.overlay = overlay
        self.dim = dim
        self.n_samples = n_samples
        self.dma = overlay.axi_dma_0
        self.brams = []
        self.brams_bit = int(np.math.ceil(np.log2(n_brams)))
        self.nodera_bit = int(np.math.ceil(np.log2(max_inst)))
        self.info_bit = int(max(np.math.ceil(np.log2(n_classes)),self.nodera_bit))
        self.attr_bit = int(np.math.ceil(np.log2(n_attr)))
        self.tree_bit = int(np.math.ceil(np.log2(trees_in_bram)))
        self.scores_bit = int(np.math.ceil(np.log2(n_brams*trees_in_bram)))
        self.depths_bit = int(np.math.ceil(np.log2(n_weights)))
        self.n_attr = n_attr
        self.n_classes = n_classes
        self.n_weights = n_weights
        for i in range(n_brams):
            variable_value = getattr(self.overlay,"axi_bram_ctrl_" + str(i))
            self.brams.append(variable_value)
            
        self.converter = converter
        self.frequency = frq_MHZ*(10**6)

        self.buff_in = pynq.allocate((self.n_samples,self.dim),dtype=np.uint8)
        self.buff_in[:] = np.zeros((self.n_samples,self.dim),dtype=np.uint8)
        self.buff_out = pynq.allocate((self.n_samples,self.dim),dtype=np.uint8)
        self.buff_out[:] = np.zeros((self.n_samples,self.dim),dtype=np.uint8)
        self.platform = platform
    
    def init_bram(self,bram,node_instructions):
        total_bits = 32 + self.attr_bit + 2*self.info_bit + 3 + self.depths_bit
        if total_bits <= 64:
            for i,node_instr in enumerate(node_instructions):  
                uint_result = self.converter.forward_conversion(input_data=node_instr.threshold, signed=True, total_bits=16, fractional_bits=8)
                first_byte = int(uint_result) | 0 << 16
                #print(type(node_instr.rctype),type(node_instr.lctype),type(node_instr.rcinfo),type(node_instr.lcinfo),type(node_instr.attr_id),type(node_instr.nodeRA))
                #print("depth_indicator: shift " + str((self.attr_bit + 2*self.info_bit + 3)) + ", data=" + str(node_instr.depth_indicator))
                #print("is_valid: shift " + str((self.attr_bit + 2*self.info_bit + 2)) + ", data=" + str(node_instr.is_valid))
                #print("rctype: shift " + str((self.attr_bit + 2*self.info_bit + 1)) + ", data=" + str(node_instr.rctype))
                #print("lctype: shift " + str(self.attr_bit + 2*self.info_bit) + ", data=" + str(node_instr.lctype))
                #print("rcinfo: shift " + str(self.attr_bit + self.info_bit) + ", data=" + str(node_instr.rcinfo))
                #print("lcinfo: shift " + str(self.attr_bit) + ", data=" + str(node_instr.lcinfo))
                #print("attr id: " + str(node_instr.attr_id) + " shift 0")
                second_byte = (node_instr.depth_indicator << (self.attr_bit + 2*self.info_bit + 3)) | \
                              (node_instr.is_valid << (self.attr_bit + 2*self.info_bit + 2)) | \
                              (node_instr.rctype << (self.attr_bit + 2*self.info_bit + 1)) | \
                              (node_instr.lctype << (self.attr_bit + 2*self.info_bit)) | \
                              (node_instr.rcinfo << (self.attr_bit + self.info_bit)) | \
                              (node_instr.lcinfo << (self.attr_bit) ) | node_instr.attr_id
                
                
                bram.write(node_instr.nodeRA*8, first_byte)
                bram.write(node_instr.nodeRA*8 + 4, second_byte)
                #print("Writing " + str(format(second_byte,'032b')) + " at " + str(node_instr.nodeRA*8))
                #print("Writing " + str(format(second_byte,'032b')) + " at " + str(node_instr.nodeRA*8 + 4))
                #print("NOINST: ", first_byte | (second_byte << 32))
                #print(first_byte, second_byte)
                
                '''
                #old version
                first_byte = int(uint_result)
                second_byte = (node_instr.rctype << self.nodera_bit + self.attr_bit + 2*self.info_bit + self.brams_bit) | \
                              (node_instr.lctype << self.nodera_bit + self.attr_bit + 2*self.info_bit) | \
                              (node_instr.rcinfo << self.nodera_bit + self.attr_bit + self.info_bit) | \
                              (node_instr.lcinfo << self.nodera_bit + self.attr_bit ) | \
                              (node_instr.attr_id << self.nodera_bit) | node_instr.nodeRA
                bram.write(node_instr.nodeRA*8, first_byte)
                bram.write(node_instr.nodeRA*8 + 4, second_byte)
                '''
                
        else:
            print("ERRORE")
            for i,node_instr in enumerate(node_instructions):  
                uint_result = self.converter.forward_conversion(input_data=node_instr.threshold, signed=True, total_bits=32, fractional_bits=16)
                first_byte = struct.pack('I',uint_result)
                bram.write(i*16, first_byte)
                bits = [self.nodera_bit,self.attr_bit,self.info_bit,self.info_bit,self.brams_bit,self.brams_bit,0]
                values = [node_instr.nodeRA,node_instr.attr_id,node_instr.lcinfo,node_instr.rcinfo,node_instr.lctype,node_instr.rctype]
                j=0
                second_byte = 0
                while np.sum(bits[:j+1])<=32:
                    second_byte = second_byte | (values[j] << int(np.sum(bits[:j])))
                    j += 1
                to_fill = 32 - np.sum(bits[:j])
                binary_value = bin(values[j])[2:].zfill(bits[j])
                second_byte = second_byte | (int(binary_value[-to_fill:],2) << int(np.sum(bits[:j])))
                bram.write(i*16 + 4, second_byte)
                third_byte = int(binary_value[:-to_fill],2)
                j += 1
                while(j<6):
                    third_byte = third_byte | (values[j] << int(np.sum(bits[:j]) - 32))
                    j += 1
                bram.write(i*16 + 8, third_byte)
        
    def init_samples(self,samples):
        assert self.n_samples == len(samples)
        for s in tqdm(range(self.n_samples)):
            for i in range(len(samples[s].features)):
                int_value = self.converter.forward_conversion(input_data=samples[s].features[i], signed=True, total_bits=32, fractional_bits=16)
                b = struct.pack('I', int_value)
                a = list(b)
                a = [int(x) for x in a]
                for j,value in enumerate(a):
                    self.buff_in[s,(4*i+j)] = value
            self.buff_in[s,4*len(samples[s].features)+1] = samples[s].offset
            self.buff_in[s,4*len(samples[s].features)+2] = samples[s].shift
            self.buff_in[s,4*len(samples[s].features)+3] = samples[s].search_for_root
            self.buff_in[s,4*len(samples[s].features)+4] = samples[s].tree_to_exec
            
            for i in range(len(samples[s].scores)):
                self.buff_in[s,4*len(samples[s].features)+6+2*i] = samples[s].scores[i]
                
            for i in range(len(samples[s].weights)):
                int_value = self.converter.forward_conversion(input_data=samples[s].weights[i], signed=True, total_bits=16, fractional_bits=8)
                b = struct.pack('I', int_value)
                a = list(b)
                a = [int(x) for x in a]
                for j,value in enumerate(a[:2]):
                    self.buff_in[s,4*len(samples[s].features)+5+2*len(samples[s].scores)+2*i+j+1] = value
        print(self.buff_in[0:2])
    
        
    def init_accel(self,samples,node_instructions_dict,flinst):
        
        for i,bram in tqdm(enumerate(self.brams)):
            self.init_bram(bram,node_instructions_dict[i])
        
        self.init_samples(samples)
        
    def exec_test(self):
        
        '''
        n_samples_per_batch = 130
            
        stop = False
        i = 0
        while not stop:
            start = n_samples_per_batch*i
            if n_samples_per_batch*(i+1) >= self.n_samples:
                end = self.n_samples
                stop = True
            else:
                end = n_samples_per_batch*(i+1)
                
            self.dma.sendchannel.transfer(self.buff_in[start:end,:])
            self.dma.recvchannel.transfer(self.buff_out[start:end,:])
            
            if i==0:
                self.dma.sendchannel.start()
                self.dma.recvchannel.start()
                
            print("waiting for send to finish")
            self.dma.sendchannel.wait()
            print("waiting for receive to finish")
            self.dma.recvchannel.wait()
            
            i += 1
        '''
        
        self.dma.sendchannel.transfer(self.buff_in)
        self.dma.recvchannel.transfer(self.buff_out)
        
        self.dma.sendchannel.start()
        self.dma.recvchannel.start()
        
        print("waiting...")
        
        self.dma.recvchannel.wait()
        
        self.buff_out.invalidate()
        outs = self.buff_out
        samples = []
        clock_cycles = []
        for out in outs:
            features = []
            for i in range(self.n_attr):
                int_result = out[i*4]*(2**0)+out[i*4+1]*(2**8)+out[i*4+2]*(2**16)+out[i*4+3]*(2**24)
                features.append(self.converter.backward_conversion(input_data=int_result, total_bits=32, fractional_bits=16)[0])
            scores = []
            for i in range(self.n_classes):
                int_result = out[4*self.n_attr+6+2*i]*(2**0)+out[4*self.n_attr+6+2*i+1]*(2**8)
                scores.append(self.converter.backward_conversion(input_data=int_result, total_bits=16, fractional_bits=8)[0])
            weights = np.zeros(self.n_weights)
            
            cycles = 0
            for i in range(4):
                cycles += out[4*self.n_attr+6+2*self.n_classes+i]*2**(8*i)
            
            sample = Sample_w_Score(features,out[4*self.n_attr+2],out[4*self.n_attr],out[4*self.n_attr+3],out[4*self.n_attr+4],scores, weights)
            samples.append(sample)
            clock_cycles.append(cycles)
        #elapsed_time = (last_out-first_in)*10e-9
        #latency = (last_out-last_in)*10e-9
        #troughput = self.buff_out.shape[0]/((last_out-first_out)*10e-9)
        throguhput = self.frequency*self.n_samples/((clock_cycles[-1]-clock_cycles[0]))
        print(self.buff_out)
        del self.buff_out
        del self.buff_in
        return samples, throguhput, outs, clock_cycles#, troughput

In [ ]:
class Accel16: # 16 bits attributes version

    def __init__(self, overlay, converter, platform='Ultra96_v2', n_brams=100, n_samples=1000, dim = 1000, n_classes=3, n_attr=5, n_weights=5, trees_in_bram=3, max_inst=576, frq_MHZ = 100):
        self.overlay = overlay
        self.dim = dim
        self.n_samples = n_samples
        self.dma = overlay.axi_dma_0
        #self.timer_0 = overlay.axi_timer_0
        #self.timer_1 = overlay.axi_timer_1
        #self.gpio = overlay.axi_gpio_0
        self.brams = []
        self.brams_bit = int(np.math.ceil(np.log2(n_brams)))
        self.nodera_bit = int(np.math.ceil(np.log2(max_inst)))
        self.info_bit = int(max(np.math.ceil(np.log2(n_classes)),self.nodera_bit))
        self.attr_bit = int(np.math.ceil(np.log2(n_attr)))
        self.tree_bit = int(np.math.ceil(np.log2(trees_in_bram)))
        self.scores_bit = int(np.math.ceil(np.log2(n_brams*trees_in_bram)))
        self.depths_bit = int(np.math.ceil(np.log2(n_weights)))
        self.n_attr = n_attr
        self.n_classes = n_classes
        self.n_weights = n_weights
        for i in range(n_brams):
            variable_value = getattr(self.overlay,"axi_bram_ctrl_" + str(i))
            self.brams.append(variable_value)
            
        self.converter = converter
        self.frequency = frq_MHZ*(10**6)

        self.buff_in = pynq.allocate((self.n_samples,self.dim),dtype=np.uint8)
        self.buff_in[:] = np.zeros((self.n_samples,self.dim),dtype=np.uint8)
        self.buff_out = pynq.allocate((self.n_samples,self.dim),dtype=np.uint8)
        self.buff_out[:] = np.zeros((self.n_samples,self.dim),dtype=np.uint8)
        self.platform = platform
    
    def init_bram(self,bram,node_instructions):
        total_bits = 32 + self.attr_bit + 2*self.info_bit + 3 + self.depths_bit
        if total_bits <= 64:
            for i,node_instr in enumerate(node_instructions):  
                uint_result = self.converter.forward_conversion(input_data=node_instr.threshold, signed=True, total_bits=16, fractional_bits=8)
                first_byte = int(uint_result) | 0 << 16
                #print(type(node_instr.rctype),type(node_instr.lctype),type(node_instr.rcinfo),type(node_instr.lcinfo),type(node_instr.attr_id),type(node_instr.nodeRA))
                #print("depth_indicator: shift " + str((self.attr_bit + 2*self.info_bit + 3)) + ", data=" + str(node_instr.depth_indicator))
                #print("is_valid: shift " + str((self.attr_bit + 2*self.info_bit + 2)) + ", data=" + str(node_instr.is_valid))
                #print("rctype: shift " + str((self.attr_bit + 2*self.info_bit + 1)) + ", data=" + str(node_instr.rctype))
                #print("lctype: shift " + str(self.attr_bit + 2*self.info_bit) + ", data=" + str(node_instr.lctype))
                #print("rcinfo: shift " + str(self.attr_bit + self.info_bit) + ", data=" + str(node_instr.rcinfo))
                #print("lcinfo: shift " + str(self.attr_bit) + ", data=" + str(node_instr.lcinfo))
                #print("attr id: " + str(node_instr.attr_id) + " shift 0")
                second_byte = (node_instr.depth_indicator << (self.attr_bit + 2*self.info_bit + 3)) | \
                              (node_instr.is_valid << (self.attr_bit + 2*self.info_bit + 2)) | \
                              (node_instr.rctype << (self.attr_bit + 2*self.info_bit + 1)) | \
                              (node_instr.lctype << (self.attr_bit + 2*self.info_bit)) | \
                              (node_instr.rcinfo << (self.attr_bit + self.info_bit)) | \
                              (node_instr.lcinfo << (self.attr_bit) ) | node_instr.attr_id
                
                
                bram.write(node_instr.nodeRA*8, first_byte)
                bram.write(node_instr.nodeRA*8 + 4, second_byte)
                #print("Writing " + str(format(second_byte,'032b')) + " at " + str(node_instr.nodeRA*8))
                #print("Writing " + str(format(second_byte,'032b')) + " at " + str(node_instr.nodeRA*8 + 4))
                #print("NOINST: ", first_byte | (second_byte << 32))
                #print(first_byte, second_byte)
                
                '''
                #old version
                first_byte = int(uint_result)
                second_byte = (node_instr.rctype << self.nodera_bit + self.attr_bit + 2*self.info_bit + self.brams_bit) | \
                              (node_instr.lctype << self.nodera_bit + self.attr_bit + 2*self.info_bit) | \
                              (node_instr.rcinfo << self.nodera_bit + self.attr_bit + self.info_bit) | \
                              (node_instr.lcinfo << self.nodera_bit + self.attr_bit ) | \
                              (node_instr.attr_id << self.nodera_bit) | node_instr.nodeRA
                bram.write(node_instr.nodeRA*8, first_byte)
                bram.write(node_instr.nodeRA*8 + 4, second_byte)
                '''
                
        else:
            print("ERRORE")
            for i,node_instr in enumerate(node_instructions):  
                uint_result = self.converter.forward_conversion(input_data=node_instr.threshold, signed=True, total_bits=32, fractional_bits=16)
                first_byte = struct.pack('I',uint_result)
                bram.write(i*16, first_byte)
                bits = [self.nodera_bit,self.attr_bit,self.info_bit,self.info_bit,self.brams_bit,self.brams_bit,0]
                values = [node_instr.nodeRA,node_instr.attr_id,node_instr.lcinfo,node_instr.rcinfo,node_instr.lctype,node_instr.rctype]
                j=0
                second_byte = 0
                while np.sum(bits[:j+1])<=32:
                    second_byte = second_byte | (values[j] << int(np.sum(bits[:j])))
                    j += 1
                to_fill = 32 - np.sum(bits[:j])
                binary_value = bin(values[j])[2:].zfill(bits[j])
                second_byte = second_byte | (int(binary_value[-to_fill:],2) << int(np.sum(bits[:j])))
                bram.write(i*16 + 4, second_byte)
                third_byte = int(binary_value[:-to_fill],2)
                j += 1
                while(j<6):
                    third_byte = third_byte | (values[j] << int(np.sum(bits[:j]) - 32))
                    j += 1
                bram.write(i*16 + 8, third_byte)
        
    def init_samples(self,samples):
        assert self.n_samples == len(samples)
        for s in tqdm(range(self.n_samples)):
            for i in range(len(samples[s].features)):
                int_value = self.converter.forward_conversion(input_data=samples[s].features[i], signed=True, total_bits=32, fractional_bits=16)
                b = struct.pack('I', int_value)
                a = list(b)
                a = [int(x) for x in a]
                for j,value in enumerate(a):
                    self.buff_in[s,(2*i+j)] = value
            self.buff_in[s,2*len(samples[s].features)+1] = samples[s].offset
            self.buff_in[s,2*len(samples[s].features)+2] = samples[s].shift
            self.buff_in[s,2*len(samples[s].features)+3] = samples[s].search_for_root
            self.buff_in[s,2*len(samples[s].features)+4] = samples[s].tree_to_exec
            
            for i in range(len(samples[s].scores)):
                self.buff_in[s,2*len(samples[s].features)+6+2*i] = samples[s].scores[i]
                
            for i in range(len(samples[s].weights)):
                int_value = self.converter.forward_conversion(input_data=samples[s].weights[i], signed=True, total_bits=16, fractional_bits=8)
                b = struct.pack('I', int_value)
                a = list(b)
                a = [int(x) for x in a]
                for j,value in enumerate(a[:2]):
                    self.buff_in[s,2*len(samples[s].features)+5+2*len(samples[s].scores)+2*i+j+1] = value
        print(self.buff_in[0:2])
    
        
    def init_accel(self,samples,node_instructions_dict,flinst):
        
        for i,bram in tqdm(enumerate(self.brams)):
            self.init_bram(bram,node_instructions_dict[i])
        
        self.init_samples(samples)
        
    def exec_test(self):
        
        self.dma.sendchannel.transfer(self.buff_in)
        
        self.dma.recvchannel.transfer(self.buff_out)
        
        self.dma.sendchannel.start()
        self.dma.recvchannel.start()
        print("waiting...")
        
        self.dma.recvchannel.wait()
        
        self.buff_out.invalidate()
        outs = self.buff_out
        samples = []
        clock_cycles = []
        
        for out in outs:
            features = []
            for i in range(self.n_attr):
                int_result = out[i*2]*(2**0)+out[i*2+1]*(2**8)
                features.append(self.converter.backward_conversion(input_data=int_result, total_bits=16, fractional_bits=8)[0])
            scores = []
            for i in range(self.n_classes):
                int_result = out[2*self.n_attr+6+2*i]*(2**0)+out[2*self.n_attr+6+2*i+1]*(2**8)
                scores.append(self.converter.backward_conversion(input_data=int_result, total_bits=16, fractional_bits=8)[0])
            weights = np.zeros(self.n_weights)
            
            cycles = 0
            for i in range(4):
                cycles += out[2*self.n_attr+6+2*self.n_classes+i]*2**(8*i)
            
            sample = Sample_w_Score(features,out[2*self.n_attr+2],out[2*self.n_attr],out[2*self.n_attr+3],out[2*self.n_attr+4],scores, weights)
            samples.append(sample)
            clock_cycles.append(cycles)
        
        #elapsed_time = (last_out-first_in)*10e-9
        #latency = (last_out-last_in)*10e-9
        #troughput = self.buff_out.shape[0]/((last_out-first_out)*10e-9)
        throguhput = self.frequency*self.n_samples/((clock_cycles[-1]-clock_cycles[0]))
        print(self.buff_out)
        del self.buff_out
        del self.buff_in
        return samples, throguhput, outs, clock_cycles#, troughput

In [ ]:
#Single execution. Insert the deploy folder into the deploys folder
overlay = "./Deploys/DeployParametricDepth5Trees10Frq166Paths5Attr5/design_2_wrapper.bit"

test = pynq.Overlay(overlay,download=True)
seed = 1234
frq = 166
dim = 4*n_attr + 2*(n_classes + n_depths) + 6 
while(dim%4 != 0):
    dim += 1

print(dim)
np.random.seed(seed)
shift = 0
offset = 0
search_for_root = 1
tree_to_exec = 0
scores = np.zeros(n_classes)
instances = []
weights = [1,1,1,1,1]
n_samples = X_test_reduced.shape[0]
for i in range(X_test_reduced.shape[0]):#n_samples):
    sample = Sample_w_Score(X_test_reduced[i%X_test_reduced.shape[0],:n_attr], shift, 0, search_for_root, 0, scores, weights)
    instances.append(sample)

node_instructions_dict = {}
j = -1
end = False
counter = 0
for i in range(len(NOInsts_list)):

    NOInsts = NOInsts_list[i].copy()

    while len(NOInsts) != 0:
        signal = False
        for i in range(max_trees_per_set):
            #print(node_instructions_dict)
            inst = NOInsts.pop(0)
            if i==0:
                exit = False
                layer = -1
                while(not exit):
                    if inst.layer != layer:
                        j += 1
                        layer = inst.layer
                        node_instructions_dict[j] = []
                    node_instructions_dict[j].append(NOInst(int(inst.nodeRA),int(inst.attr_id),inst.threshold,int(inst.lctype),int(inst.lcinfo),int(inst.rctype),int(inst.rcinfo),inst.layer,inst.is_valid,inst.depth_indicator))
                    if len(NOInsts) == 0:
                        exit=True
                        signal = True
                    elif NOInsts[0].layer==0:
                        exit = True
                    else:
                        inst = NOInsts.pop(0)
            else:
                exit = False
                while(not exit):
                    node_instructions_dict[counter*max_depth+inst.layer].append(NOInst(int(inst.nodeRA),int(inst.attr_id),inst.threshold,int(inst.lctype),int(inst.lcinfo),int(inst.rctype),int(inst.rcinfo),inst.layer,inst.is_valid,inst.depth_indicator))
                    if len(NOInsts) == 0:
                        exit = True
                        signal = True
                    elif NOInsts[0].layer==0:
                        exit = True
                    else:
                        inst = NOInsts.pop(0)
            if signal:
                break
        counter += 1
        
'''
node_instructions_dict = {}
for i in range(5):
    node_instructions_dict[i] = []
for i in range(10):
    # address[16], attr_id[2], threshold[16],  lctype[1] lcinfo[10] rctype[1], rcinfo[10]
    # is_valid[1], depth_indicator[3] + 4 empty
    node_instructions_dict[0].append(NOInst(i,2**28,0,0,0,0,0,0,True,0))
    #node_instructions_dict[0].append(NOInst(i,0,127.111,0,0,0,0,0,0,0))
'''

converter = Converter()

#return samples#, latency, throughput

In [ ]:
throughputs = []
skip = False
final = 0
for i in tqdm(range(1)):
    test = pynq.Overlay(overlay,download=True)
    accel = Accel(test, converter, n_brams = n_topfs*n_paths, n_samples = len(instances), dim = dim, n_classes = n_classes, n_weights = len(weights), trees_in_bram = max_trees_per_set, n_attr = n_attr, max_inst = max_inst,frq_MHZ = frq)
    accel.init_accel(instances,node_instructions_dict,FLInsts_list[0])
    samples, throughput, buff, clock_cycles = accel.exec_test()
    if throughputs.count(throughput) >= 2:
        skip = True
        final = throughput

    throughputs.append(throughput)

    if skip:
        break

#if not skip:
#    throughputs = np.sort(throughputs)
#    thr = np.mean(throughputs[2:9])/1e6
#else:
#    thr = final

In [ ]:
samples[0].scores

In [ ]:
#function to execute multiple times and compute average throughput if necessary
def main(total_pes):
    
    seed = 1234
    dim = 4*n_attr + 2*(n_classes + n_depths) + 6 
    while(dim%4 != 0):
        dim += 1
        
    print("dim: ", dim)

    np.random.seed(seed)
    shift = 0
    offset = 0
    search_for_root = 1
    tree_to_exec = 0
    scores = np.zeros(n_classes)
    instances = []
    weights = np.ones(n_depths)
    n_samples = 132 #X_test_reduced.shape[0]
    for i in range(n_samples):
        sample = Sample_w_Score(X_test_reduced[i%X_test_reduced.shape[0],:n_attr], shift, 0, search_for_root, 0, scores, weights)
        instances.append(sample)

    node_instructions_dict = {}
    j = -1
    end = False
    counter = 0
    
    for i in range(len(NOInsts_list)):

        NOInsts = NOInsts_list[i].copy()

        while len(NOInsts) != 0:
            signal = False
            for k in range(max_trees_per_set):
                inst = NOInsts.pop(0)
                if k==0:
                    exit = False
                    layer = -1
                    while(not exit):
                        if inst.layer != layer:
                            j += 1
                            layer = inst.layer
                            node_instructions_dict[j] = []
                        node_instructions_dict[j].append(NOInst(int(inst.nodeRA),int(inst.attr_id),inst.threshold,int(inst.lctype),int(inst.lcinfo),int(inst.rctype),int(inst.rcinfo),inst.layer,inst.is_valid,inst.depth_indicator))
                        if len(NOInsts) == 0:
                            exit=True
                            signal = True
                        elif NOInsts[0].layer==0:
                            exit = True
                        else:
                            inst = NOInsts.pop(0)
                else:
                    exit = False
                    while(not exit):
                        node_instructions_dict[counter*max_depth+inst.layer].append(NOInst(int(inst.nodeRA),int(inst.attr_id),inst.threshold,int(inst.lctype),int(inst.lcinfo),int(inst.rctype),int(inst.rcinfo),inst.layer,inst.is_valid,inst.depth_indicator))
                        if len(NOInsts) == 0:
                            exit = True
                            signal = True
                        elif NOInsts[0].layer==0:
                            exit = True
                        else:
                            inst = NOInsts.pop(0)
                if signal:
                    break
            counter += 1
            
    converter = Converter()
    
    throughputs = []
    skip = False
    final = 0
    for i in tqdm(range(15)):
        test = pynq.Overlay(overlay,download=True)
        print(total_pes)
        accel = Accel(test, converter, n_brams = total_pes, n_samples = len(instances), dim = dim, n_classes = n_classes, n_weights = len(weights), trees_in_bram = max_trees_per_set, n_attr = n_attr, max_inst = max_inst,frq_MHZ = frq)
        accel.init_accel(instances,node_instructions_dict,FLInsts_list[0])
        samples, throughput, buff, clock_cycles = accel.exec_test()
        print(throughput)
        if throughputs.count(throughput) >= 1:
            skip = True
            final = throughput

        throughputs.append(throughput)

        if skip:
            break

    if not skip:
        "correct throughput not found, returning average of all values"
        throughputs = np.sort(throughputs)
        thr = np.mean(throughputs[2:9])/1e6
        clock_cycles = []
    else:
        "correct throughput found"
        thr = final
        
    return samples, thr, clock_cycles 

In [ ]:
string = ''

In [ ]:
root_folder = './Deploys'
folders = os.listdir(root_folder)
folders.sort()
folders

In [ ]:
folders = os.listdir(root_folder)
folders.sort()
throughputs = []
clock_cycles_list = []
samples_list = []
for folder in folders:
    overlay = root_folder + folder + '/design_2_wrapper.bit'
    print("Executing project: " + folder)
    folderParams = re.findall(r'\d+', folder)
    n_trees = int(folderParams[1])
    max_depth = int(folderParams[0])
    frq = int(folderParams[2])
    n_paths = int(folderParams[3])
    n_attr = int(folderParams[4])
    min_depth = max_depth - 4
    n_depths = max_depth - min_depth + 1
    n_classes = 6
    X_train_reduced = X_train[Y_train<n_classes]
    Y_train_reduced = Y_train[Y_train<n_classes]
    X_test_reduced = X_test[Y_test<n_classes]
    Y_test_reduced = Y_test[Y_test<n_classes]
    print("selected number of trees", n_trees)
    if n_trees < n_paths*n_depths:
        updated_n_trees = n_paths*n_depths
    else:
        updated_n_trees = n_trees
        
    print("trained number of trees", updated_n_trees)
    
    mdcls = MultiDepthNeuralRandomForestClassifier(total_estimators=updated_n_trees,min_depth=min_depth,max_depth=max_depth,random_state=3)
    mdcls.fit(X_train_reduced[:,:n_attr], Y_train_reduced)
    print("model accuracy: ", mdcls.score(X_test_reduced[:,:n_attr],Y_test_reduced))
    
    bram_size = 36*1024
    instruction_per_bram = int(bram_size/64)
    trees_per_path = int(updated_n_trees/n_paths) #assume at the moment that the total number of trees is multiple of n_paths
    trees_per_path_per_depth = int(trees_per_path/n_depths)
    max_trees_per_set = instruction_per_bram//((2**(max_depth-1)))*n_depths
    necessary_set_of_pes = int(math.ceil(updated_n_trees/max_trees_per_set))
    
    max_inst = 576
    NOInsts_list = []
    FLInsts_list = []
    trees_per_depth = int(updated_n_trees/n_depths)
    unaligned_trees = (trees_per_depth%n_paths)*n_depths
    iteration_per_depth = {}
    added_per_depth = {}
    for k in range(n_depths):
        iteration_per_depth[k] = 0
        added_per_depth[k] = 0
    
    for path in range(n_paths):
        additional_trees = math.ceil(unaligned_trees/(n_paths - path))
        unaligned_trees -= additional_trees
        estimators = []
        for k in range(n_depths):
            to_be_added = min(additional_trees,trees_per_depth%n_paths-added_per_depth[k])
            estimators.append(list(mdcls.classifiers.values())[k].estimators_[iteration_per_depth[k]:iteration_per_depth[k]+trees_per_path_per_depth + to_be_added])
            iteration_per_depth[k] += trees_per_path_per_depth + to_be_added
            added_per_depth[k] += to_be_added
            additional_trees -= to_be_added
        
        FLInsts_per_path, NOInsts_per_depth = transform_into_instructions(estimators,max_inst,1000000,max_trees_per_set,n_depths)
        NOInsts_list.append(NOInsts_per_depth)
        FLInsts_list.append(FLInsts_per_path)

    if necessary_set_of_pes >= n_paths:
        total_pes = max_depth*necessary_set_of_pes
    else:
        total_pes = max_depth*n_paths

    #dic = main(total_pes)
    samples, throughput, clock_cycles = main(total_pes)
    
    samples_list.append(samples)
    throughputs.append(throughput)
    clock_cycles_list.append(clock_cycles)
    string += folder + ": " + "Throughput " + str(throughput/1e6)
    string += " \n"
    print(folder + ": " + "Throughput " + str(throughput/1e6))
    

In [ ]:
n_trees_list = []
max_depths_list = []
n_paths_list = []
n_attrs_list = []
for folder in folders:
    #overlay = '/home/xilinx/pynq/YoseUeSATL/DeploysMultiPath/' + folder + '/design_2_wrapper.bit'
    #print("Executing project: " + folder)
    folderParams = re.findall(r'\d+', folder)
    n_trees = int(folderParams[1])
    max_depth = int(folderParams[0])
    frq = int(folderParams[2])
    n_paths = int(folderParams[3])
    n_attr = int(folderParams[4])
    n_trees_list.append(n_trees)
    max_depths_list.append(max_depth)
    n_paths_list.append(n_paths)
    n_attrs_list.append(n_attr)
    
df_dict = {}
df_dict['n_trees'] = n_trees_list
df_dict['max_depth'] = max_depths_list
df_dict['n_attr'] = n_attrs_list
df_dict['n_paths'] = n_paths_list
df_dict['thorughput'] = throughputs
df_dict['clock_cycles'] = clock_cycles_list

In [ ]:
df = pd.DataFrame.from_dict(df_dict)

In [ ]:
avg_latencies = []
for i in range(len(df)):
    latencies = []
    cycles = list(df['clock_cycles'][i]) #ast.literal_eval(df['clock_cycles'][i])
    for j in range(0,130):
        latencies.append(cycles[j]-3*j)
    avg_latencies.append(np.mean(latencies))
    
exec_time = []
for i in range(len(df)):
    cycles = list(df['clock_cycles'][i])[-1] #ast.literal_eval(df['clock_cycles'][i])[-1]
    exec_time.append(cycles)
    
df['avg_latency'] = avg_latencies
df['exec_time'] = exec_time

In [ ]:
df.to_csv('./results/singlepath_balanced.csv')

## Uneven paths case

In [ ]:
root_folder = './Deploys/'
folders = os.listdir(root_folder)
folders.sort()
folders

In [ ]:
folders = os.listdir(root_folder)
folders.sort()
throughputs = []
clock_cycles_list = []
samples_list = []
for folder in folders:
    overlay = root_folder + folder + '/design_2_wrapper.bit'
    print("Executing project: " + folder)
    folderParams = re.findall(r'\d+', folder)
    n_trees = int(folderParams[1])
    max_depth = int(folderParams[0])
    frq = int(folderParams[2])
    n_paths = int(folderParams[3])
    n_attr = int(folderParams[4])
    min_depth = max_depth - 4
    n_depths = max_depth - min_depth + 1
    n_classes = 6
    X_train_reduced = X_train[Y_train<n_classes]
    Y_train_reduced = Y_train[Y_train<n_classes]
    X_test_reduced = X_test[Y_test<n_classes]
    Y_test_reduced = Y_test[Y_test<n_classes]
    print("selected number of trees", n_trees)
    if n_trees < n_paths*n_depths:
        updated_n_trees = n_paths*n_depths
    else:
        updated_n_trees = n_trees
        
    print("trained number of trees", updated_n_trees)
    
    mdcls = MultiDepthNeuralRandomForestClassifier(total_estimators=updated_n_trees,min_depth=min_depth,max_depth=max_depth,random_state=3)
    mdcls.fit(X_train_reduced[:,:n_attr], Y_train_reduced)
    print("model accuracy: ", mdcls.score(X_test_reduced[:,:n_attr],Y_test_reduced))
    
    bram_size = 36*1024
    instruction_per_bram = int(bram_size/64)
    trees_per_path = int(updated_n_trees/n_paths) #assume at the moment that the total number of trees is multiple of n_paths
    trees_per_path_per_depth = int(trees_per_path/n_depths)
    max_trees_per_set = instruction_per_bram//((2**(max_depth-1)))*n_depths
    necessary_set_of_pes = int(math.ceil(updated_n_trees/max_trees_per_set))
    
    max_inst = 576
    NOInsts_list = []
    FLInsts_list = []
    trees_per_depth = int(updated_n_trees/n_depths)
    unaligned_trees = (trees_per_depth%n_paths)*n_depths
    iteration_per_depth = {}
    added_per_depth = {}
    for k in range(n_depths):
        iteration_per_depth[k] = 0
        added_per_depth[k] = 0
    
    for path in range(n_paths):
        additional_trees = math.ceil(unaligned_trees/(n_paths - path))
        unaligned_trees -= additional_trees
        estimators = []
        for k in range(n_depths):
            to_be_added = min(additional_trees,trees_per_depth%n_paths-added_per_depth[k])
            estimators.append(list(mdcls.classifiers.values())[k].estimators_[iteration_per_depth[k]:iteration_per_depth[k]+trees_per_path_per_depth + to_be_added])
            iteration_per_depth[k] += trees_per_path_per_depth + to_be_added
            added_per_depth[k] += to_be_added
            additional_trees -= to_be_added
        
        FLInsts_per_path, NOInsts_per_depth = transform_into_instructions(estimators,max_inst,1000000,max_trees_per_set,n_depths)
        NOInsts_list.append(NOInsts_per_depth)
        FLInsts_list.append(FLInsts_per_path)

    if max_depth == 5:
        total_pes = max_depth*(n_paths - 1) - 2
    else:
        total_pes = max_depth*(n_paths - 1)

    #dic = main(total_pes)
    samples, throughput, clock_cycles = main(total_pes)
    
    samples_list.append(samples)
    throughputs.append(throughput)
    clock_cycles_list.append(clock_cycles)
    string += folder + ": " + "Throughput " + str(throughput/1e6)
    string += " \n"
    print(folder + ": " + "Throughput " + str(throughput/1e6))
    

In [ ]:
n_trees_list = []
max_depths_list = []
n_paths_list = []
n_attrs_list = []
for folder in folders:
    #overlay = '/home/xilinx/pynq/YoseUeSATL/DeploysMultiPath/' + folder + '/design_2_wrapper.bit'
    #print("Executing project: " + folder)
    folderParams = re.findall(r'\d+', folder)
    n_trees = int(folderParams[1])
    max_depth = int(folderParams[0])
    frq = int(folderParams[2])
    n_paths = int(folderParams[3])
    n_attr = int(folderParams[4])
    n_trees_list.append(n_trees)
    max_depths_list.append(max_depth)
    n_paths_list.append(n_paths)
    n_attrs_list.append(n_attr)
    
df_dict = {}
df_dict['n_trees'] = n_trees_list
df_dict['max_depth'] = max_depths_list
df_dict['n_attr'] = n_attrs_list
df_dict['n_paths'] = n_paths_list
df_dict['thorughput'] = throughputs
df_dict['clock_cycles'] = clock_cycles_list

In [ ]:
df = pd.DataFrame.from_dict(df_dict)
avg_latencies = []
for i in range(len(df)):
    latencies = []
    cycles = list(df['clock_cycles'][i]) #ast.literal_eval(df['clock_cycles'][i])
    for j in range(0,130):
        latencies.append(cycles[j]-3*j)
    avg_latencies.append(np.mean(latencies))
    
exec_time = []
for i in range(len(df)):
    cycles = list(df['clock_cycles'][i])[-1] #ast.literal_eval(df['clock_cycles'][i])[-1]
    exec_time.append(cycles)
    
df['avg_latency'] = avg_latencies
df['exec_time'] = exec_time

In [ ]:
df